# Galaxy Redshift Prediction with Physically Informed Late Fusion: Band-Aware EfficientNet + Band-Aware Transformer


## 1. Setup

In [1]:
import os, sys, csv, copy, random, time
from pathlib import Path
import contextlib

project_root = str(Path(os.getcwd()).parent)
sys.path.insert(0, project_root)

import numpy as np
import pandas as pd
import h5py

import torch
import torch.nn as nn
import torch.optim as optim
from torch import autocast
from torchvision.models import efficientnet_b0, EfficientNet_B0_Weights
from torch.optim.lr_scheduler import CosineAnnealingLR, LinearLR, SequentialLR

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
from IPython.display import clear_output
from scipy.stats import norm as scipy_norm
from scipy.stats import gaussian_kde
from tqdm.notebook import tqdm

from src.data_loader import (BAND_MEAN, BAND_STD, MORPH_COLS, MAG_COLS, TABULAR_COLS,
                              PhysFusionGalaxyDataset, get_phys_fusion_dataloaders)
from src.models.phys_informed import (
    BandEmbedding, BandAwareImageBranch, BandAwareMorphologyBranch,
    GaussianHead, BandAlignedFusion, PhysicallyInformedLateFusionNet, ProbabilisticRedshiftLoss
)
from src.utils import *
from src.metrics import *

In [5]:
set_seed(42)
device = get_device()

Using MPS (Apple Metal GPU)


In [6]:
BASE_DATA = os.path.abspath("../data/")
BASE_MODELS = os.path.abspath("../models/")
BASE_REPORTS = os.path.abspath("../reports/")
os.makedirs(BASE_MODELS, exist_ok=True)

TRAIN_PATH = os.path.join(BASE_DATA, "5x127x127_training_with_morphology.hdf5")
VAL_PATH = str(os.path.join(BASE_DATA, "5x127x127_validation_with_morphology.hdf5"))
TEST_PATH = str(os.path.join(BASE_DATA, "5x127x127_testing_with_morphology.hdf5"))

SAVE_PATH = str(os.path.join(BASE_MODELS, "physfusion_best.pth"))
RESUME_PATH = str(os.path.join(BASE_MODELS, "physfusion_latest.pth"))
CSV_PATH = str(os.path.join(BASE_MODELS, "physfusion_metrics.csv"))

BATCH_SIZE = 64
NUM_EPOCHS = 50
EMBED_DIM = 128
MORPH_TOK_DIM = 64
WARMUP_EPOCHS = 5
BACKBONE_LR = 1e-5
HEAD_LR = 1e-3
WEIGHT_DECAY = 1e-4
ETA_MIN = 1e-6
AUX_ALPHA = 0.3
HUBER_DELTA = 0.1
ACCUM_STEPS = 4
PATIENCE = 12
NUM_WORKERS = 0

BANDS = ["g", "r", "i", "z", "y"]
N_BANDS = len(BANDS)

BAND_WAVELENGTHS = [4800., 6200., 7700., 8900., 9800.]

N_COLORS = 9
COLOR_BAND_IDX = [0, 1, 2, 3, 0, 0, 1, 1, 0]
NUM_TABULAR = len(TABULAR_COLS) + N_COLORS

FEATURE_BAND_IDX = torch.tensor(
    [0,1,2,3,4] +
    [0,1,2,3,4] +
    [0]*12+[1]*12+[2]*12+[3]*12+[4]*12 +
    COLOR_BAND_IDX,
    dtype=torch.long
) 

print(f"Stored tabular features : {len(TABULAR_COLS)}")
print(f"Colors (on-the-fly)     : {N_COLORS}")
print(f"Total features to model : {NUM_TABULAR}")
print(f"Band feature counts     : { {b: (FEATURE_BAND_IDX==i).sum().item() for i,b in enumerate(BANDS)} }")

Stored tabular features : 70
Colors (on-the-fly)     : 9
Total features to model : 79
Band feature counts     : {'g': 18, 'r': 17, 'i': 15, 'z': 15, 'y': 14}


### 1.1 GPU-specific settings

In [7]:
if device.type == 'cuda':
    torch.backends.cudnn.benchmark = True

    torch.set_float32_matmul_precision('high')

    props = torch.cuda.get_device_properties(device)
    print(f'  GPU  : {props.name}')
    print(f'  VRAM : {props.total_memory / 1e9:.1f} GB')
    print(f'  SMs  : {props.multi_processor_count}')
    NUM_WORKERS = 0
                      
print(f'  NUM_WORKERS : {NUM_WORKERS}')


  NUM_WORKERS : 0


## 2. Dataset

In [8]:
train_loader, val_loader, test_loader, tab_mean, tab_std = get_phys_fusion_dataloaders(
    TRAIN_PATH, VAL_PATH, TEST_PATH,
    batch_size=BATCH_SIZE, num_workers=NUM_WORKERS, pin_memory=device.type == "cuda",
)

np.savez(str(os.path.join(BASE_MODELS, 'physfusion_tabular_norm.npz')), mean=tab_mean, std=tab_std)
print('Tabular norm stats saved.')

imgs, tabs, labels = next(iter(train_loader))
print(f'images  : {imgs.shape}   NaN={imgs.isnan().any().item()}')
print(f'tabular : {tabs.shape}  NaN={tabs.isnan().any().item()}')
print(f'labels  : min={labels.min():.3f}  max={labels.max():.3f}  mean={labels.mean():.3f}')
print(f'DataLoader  workers={NUM_WORKERS}  pin_memory={device.type == "cuda"}')

Loaded 5x127x127_training_with_morphology.hdf5: 204,573 samples
Loaded 5x127x127_validation_with_morphology.hdf5: 40,914 samples
Loaded 5x127x127_testing_with_morphology.hdf5: 40,914 samples
Tabular norm stats saved.
images  : torch.Size([64, 5, 127, 127])   NaN=False
tabular : torch.Size([64, 79])  NaN=False
labels  : min=0.045  max=2.756  mean=0.588
DataLoader  workers=0  pin_memory=False


## 3. Model

In [9]:
model = PhysicallyInformedLateFusionNet(
    num_tabular=NUM_TABULAR,
    feature_band_idx=FEATURE_BAND_IDX,
    embed_dim=EMBED_DIM,
    morph_token_dim=MORPH_TOK_DIM,
    morph_heads=4,
    morph_layers=3,
    fusion_heads=4,
    fusion_layers=2,
    dropout=0.1,
    img_pretrained=True,
    img_freeze=False,
).to(device)


if device.type == 'cuda' and sys.platform != 'win32':
    model = model.to(memory_format=torch.channels_last)
    print('Model moved to channels_last memory format.')
else:
    print('Using contiguous memory format (Windows or non-CUDA).')

def _try_compile(m):
    try:
        import triton
        compiled = torch.compile(m)
        print('Model compiled with torch.compile (Triton backend).')
        return compiled
    except ImportError:
        print('Triton not available (expected on Windows) — using eager mode.')
        return m
    except Exception as e:
        print(f'torch.compile skipped ({e.__class__.__name__}) — using eager mode.')
        return m

if device.type == 'cuda' and hasattr(torch, 'compile'):
    model = _try_compile(model)

total = sum(p.numel() for p in model.parameters())
trainp = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total params     : {total:,}')
print(f'Trainable params : {trainp:,}')

with torch.no_grad():
    _imgs_gpu = imgs.to(device)
    if device.type == 'cuda':
        _imgs_gpu = _imgs_gpu.to(memory_format=torch.channels_last)
    _z, _sigma, _za, _zm = model(_imgs_gpu, tabs.to(device))
print(f'z_pred shape : {_z.shape}   sigma shape : {_sigma.shape}')
print('Smoke test passed')


/opt/anaconda3/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


Using contiguous memory format (Windows or non-CUDA).
Total params     : 4,694,912
Trainable params : 4,694,912
z_pred shape : torch.Size([64])   sigma shape : torch.Size([64])
Smoke test passed


In [10]:
criterion = ProbabilisticRedshiftLoss(alpha=AUX_ALPHA, beta=0.5, huber_delta=HUBER_DELTA)

optimizer = optim.AdamW([
      {'params': model.image_branch.global_features.parameters(), 'lr': BACKBONE_LR},
      {'params': model.image_branch.global_proj.parameters(), 'lr': HEAD_LR},
      {'params': model.image_branch.band_cnn.parameters(), 'lr': HEAD_LR},
      {'params': model.image_branch.band_proj.parameters(), 'lr': HEAD_LR},
      {'params': model.morph_branch.parameters(), 'lr': HEAD_LR},
      {'params': model.fusion.parameters(), 'lr': HEAD_LR},
      {'params': model.aux_img_head.parameters(), 'lr': HEAD_LR},
      {'params': model.aux_morph_head.parameters(), 'lr': HEAD_LR},
  ], weight_decay=WEIGHT_DECAY)

warmup = LinearLR(optimizer, start_factor=0.1, end_factor=1.0, total_iters=WARMUP_EPOCHS)
cosine = CosineAnnealingLR(optimizer, T_max=max(1, NUM_EPOCHS - WARMUP_EPOCHS), eta_min=ETA_MIN)
scheduler = SequentialLR(optimizer, schedulers=[warmup, cosine], milestones=[WARMUP_EPOCHS])
scaler = None

## 4. Training 

In [11]:
checkpoint_template = {
    "model" : "PhysicallyInformedLateFusionNet",
    "architecture" : {
        "image_branch" : "BandAwareImageBranch (EfficientNet-B0 global + shared-CNN per-band × 5)",
        "morph_branch" : "BandAwareMorphologyBranch (Transformer, band-tagged tokens, per-band pool)",
        "fusion" : "BandAlignedFusion (11-token self-attention, 2 layers)",
        "band_embed" : "Shared BandEmbedding (sinusoidal wavelength init, learnable)",
    },
    "model_params" : {
        "num_tabular" : NUM_TABULAR,
        "embed_dim" : EMBED_DIM,
        "morph_token_dim" : MORPH_TOK_DIM,
        "morph_heads" : 4,
        "morph_layers" : 3,
        "fusion_heads" : 4,
        "fusion_layers" : 2,
        "dropout" : 0.1,
    },
    "gap_addressed": (
        "Physically informed fusion: band-specific attention, wavelength-tagged "
        "morphology tokens, band-aligned cross-modal self-attention."
    ),
    "criterion" : f"SmoothL1Loss(beta={HUBER_DELTA})",
    "aux_alpha" : AUX_ALPHA,
    "batch_size" : BATCH_SIZE,
    "torch_version": torch.__version__,
    "optimizer_config": {
        "backbone_lr" : BACKBONE_LR,
        "head_lr" : HEAD_LR,
        "weight_decay": WEIGHT_DECAY,
        "eta_min" : ETA_MIN,
        "T_max" : NUM_EPOCHS,
        "warmup_epochs": WARMUP_EPOCHS,
        "patience" : PATIENCE,
        "accum_steps" : ACCUM_STEPS,
    },
    "tabular_cols" : TABULAR_COLS,
    "n_colors" : N_COLORS,
    "color_band_idx": COLOR_BAND_IDX,
    "band_wavelengths": BAND_WAVELENGTHS,
    "morph_mean" : tab_mean.tolist(),
    "morph_std" : tab_std.tolist(),
    "band_mean" : BAND_MEAN.squeeze().tolist(),
    "band_std" : BAND_STD.squeeze().tolist(),
}

In [12]:
def _autocast(device_type: str, enabled: bool):
    if not enabled:
        return contextlib.nullcontext()
    try:
        return torch.amp.autocast(device_type)
    except (TypeError, AttributeError):
        return torch.cuda.amp.autocast()

def redshift_metrics(ypred: np.ndarray, ytrue: np.ndarray) -> dict:
    mask = np.isfinite(ypred) & np.isfinite(ytrue)
    ypred, ytrue = ypred[mask], ytrue[mask]

    dz = (ypred - ytrue) / (1.0 + ytrue)
    nmad = float(1.4826 * np.median(np.abs(dz - np.median(dz))))
    ssres = np.sum((ytrue - ypred) ** 2)
    sstot = np.sum((ytrue - ytrue.mean()) ** 2)
    return {
        'mae': float(np.mean(np.abs(ypred - ytrue))),
        'rmse': float(np.sqrt(np.mean((ypred - ytrue) ** 2))),
        'r2': float(1.0 - ssres / (sstot + 1e-10)),
        'nmad': nmad,
        'bias': float(np.mean(dz)),
        'outlier_pct': float(np.mean(np.abs(dz) > 0.15) * 100),
        'within_0.1z': float(np.mean(np.abs(dz) < 0.1)),
        'within_0.05z': float(np.mean(np.abs(dz) < 0.05)),
    }

def train_epoch(model, loader, optimizer, criterion, device,
                scaler=None, accum=2, beta_scale=1.0):
    model.train()
    total_loss = 0.0
    _cuda = device.type == 'cuda'
    _mem_fmt = (torch.channels_last
                   if _cuda and sys.platform != 'win32'
                   else torch.contiguous_format)
    pbar = tqdm(loader, desc='Train', leave=True)
    optimizer.zero_grad(set_to_none=True)
    adev = 'cuda' if _cuda else 'cpu'
    _total_vram = (torch.cuda.get_device_properties(device).total_memory / 1e9
                   if _cuda else 0.0)   

    for i, (imgs, tabs, targets) in enumerate(pbar):
        imgs = imgs.to(device, non_blocking=True, memory_format=_mem_fmt)
        tabs = tabs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with _autocast(adev, enabled=(scaler is not None)):
            mu, sigma, z_img, z_morph = model(imgs, tabs)

        mu = mu.float()
        sigma = sigma.float()
        z_img = z_img.float()
        z_morph = z_morph.float()
        targets = targets.float()

        loss, ld = criterion(mu, sigma, z_img, z_morph, targets,
                             beta_scale=beta_scale)
        loss = loss / accum

        if not torch.isfinite(loss):
            optimizer.zero_grad(set_to_none=True)
            continue

        if scaler: scaler.scale(loss).backward()
        else: loss.backward()

        if (i + 1) % accum == 0:
            if scaler:
                scaler.unscale_(optimizer)
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                scaler.step(optimizer); scaler.update()
            else:
                nn.utils.clip_grad_norm_(model.parameters(), 1.0)
                optimizer.step()
            optimizer.zero_grad(set_to_none=True)

        total_loss += ld['loss_total']
        gpu_gb = (f"{torch.cuda.memory_allocated(device)/1e9:.1f}/{_total_vram:.1f}GB"
                  if _cuda else 'N/A')
        pbar.set_postfix(ordered_dict={
            'NLL' : f"{ld['loss_nll']:.4f}",
            'Huber' : f"{ld['loss_huber']:.4f}",
            'avg' : f"{total_loss/(i+1):.4f}",
            'lr' : f"{optimizer.param_groups[0]['lr']:.1e}",
            'VRAM' : gpu_gb,
        })

    if (i + 1) % accum != 0:
        if scaler:
            scaler.unscale_(optimizer)
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            scaler.step(optimizer); scaler.update()
        else:
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
        optimizer.zero_grad(set_to_none=True)

    return total_loss / len(loader)

@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total_loss = 0.0
    all_mu, all_sigma, all_true = [], [], []
    _cuda = device.type == 'cuda'
    _mem_fmt = (torch.channels_last
                if _cuda and sys.platform != 'win32'
                else torch.contiguous_format)
    adev = 'cuda' if _cuda else 'cpu'

    for imgs, tabs, targets in loader:
        imgs = imgs.to(device, non_blocking=True, memory_format=_mem_fmt)
        tabs = tabs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)
        with _autocast(adev, enabled=_cuda):
            mu, sigma, zimg, zmorph = model(imgs, tabs)
            mu = mu.float().clamp(-0.1, 6.0)  # physically valid redshift range
            sigma = sigma.float().clamp(1e-4, 5.0)
            _, ld = criterion(mu, sigma, z_img, z_morph, targets, beta_scale=0.0)
        total_loss += ld['loss_total'] * imgs.size(0)
        all_mu.append(mu.cpu().float().numpy())
        all_sigma.append(sigma.cpu().float().numpy())
        all_true.append(targets.cpu().float().numpy())

    y_mu = np.concatenate(all_mu).flatten()
    y_sigma = np.concatenate(all_sigma).flatten()
    y_true = np.concatenate(all_true).flatten()

    metrics = redshift_metrics(y_mu, y_true)
    metrics['loss'] = total_loss / len(loader.dataset)
    metrics['mean_sigma'] = float(y_sigma.mean())
    metrics['med_sigma'] = float(np.median(y_sigma))

    return metrics, y_mu, y_sigma, y_true


In [13]:
train_losses = []; val_losses = []
best_loss = float("inf"); no_improve = 0; start_epoch = 0

_final_ckpt = str(os.path.join(BASE_MODELS, "physfusion_epoch_50.pth"))

if os.path.exists(_final_ckpt):
    print("Epoch 50 checkpoint detected. Model training already completed.")
elif os.path.exists(RESUME_PATH):
    ckpt = torch.load(RESUME_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt["model_state_dict"])
    optimizer.load_state_dict(ckpt["optimizer_state_dict"])
    scheduler.load_state_dict(ckpt["scheduler_state_dict"])
    if "scaler_state_dict" in ckpt and scaler and ckpt["scaler_state_dict"]:
        scaler.load_state_dict(ckpt["scaler_state_dict"])
    start_epoch = ckpt.get("epoch", 0) + 1
    best_loss = ckpt.get("best_val_loss", float("inf"))
    train_losses = ckpt.get("train_losses", [])
    val_losses = ckpt.get("val_losses", [])
    no_improve = ckpt.get("no_improve", 0)
    print(f"Resumed from epoch {start_epoch}, best_loss={best_loss:.4f}, no_improve={no_improve}")
else:
    print("No checkpoint found — starting fresh.")

if not os.path.exists(CSV_PATH):
    with open(CSV_PATH, "w", newline="") as f:
        csv.writer(f).writerow([
            "epoch", "time_s", "train_loss", "val_loss",
            "val_mae", "val_rmse", "val_r2", "val_nmad",
            "val_outlier_pct", "within_0.1z", "within_0.05z", "val_bias",
            "mean_sigma", "med_sigma",
            "gap", "best_val_loss", "no_improve", "is_best", "lr", "beta_scale",
        ])
print("CSV log ready:", CSV_PATH)

Resumed from epoch 50, best_loss=-2.5018, no_improve=5
CSV log ready: /Users/arpinejanunts/Desktop/Capstone/models/physfusion_metrics.csv


In [14]:
_diag_path = str(os.path.join(BASE_MODELS, 'physfusion_diag.txt'))

def _vram_str():
    if device.type != 'cuda': return 'N/A'
    alloc = torch.cuda.memory_allocated(device) / 1e9
    total = torch.cuda.get_device_properties(device).total_memory / 1e9
    return f'{alloc:.2f}/{total:.1f} GB'

def _log(msg):
    print(msg, flush=True)
    with open(_diag_path, 'a', encoding='utf-8') as _f:
        _f.write(msg + '\n')

with open(_diag_path, 'w', encoding='utf-8') as _f:
    _f.write(f'=== Training started ===\n')
_log(f'Diagnostic log -> {_diag_path}')

for epoch in range(start_epoch, NUM_EPOCHS):
    t0 = time.time()

    _log(f'[Epoch {epoch+1}] START  VRAM={_vram_str()}')

    beta_scale = max(0.0, 1.0 - max(0, epoch - WARMUP_EPOCHS)
                          / max(1, NUM_EPOCHS - WARMUP_EPOCHS))

    _log(f'[Epoch {epoch+1}] starting train_epoch ...')
    tr_loss = train_epoch(
        model, train_loader, optimizer, criterion,
        device, scaler, ACCUM_STEPS, beta_scale)
    _log(f'[Epoch {epoch+1}] train_epoch done  loss={tr_loss:.4f}  VRAM={_vram_str()}')

    if device.type == 'cuda':
        torch.cuda.empty_cache()
    _log(f'[Epoch {epoch+1}] cache cleared  VRAM={_vram_str()}')

    _log(f'[Epoch {epoch+1}] starting evaluate ...')
    val_metrics, val_mu, val_sigma, val_true = evaluate(
        model, val_loader, criterion, device)
    _log(f'[Epoch {epoch+1}] evaluate done  val_loss={val_metrics["loss"]:.4f}  VRAM={_vram_str()}')
    val_loss = val_metrics['loss']
    current_lr = optimizer.param_groups[0]['lr']
    scheduler.step()

    train_losses.append(tr_loss)
    val_losses.append(val_loss)
    gap = tr_loss - val_loss
    elapsed = time.time() - t0
    is_best = val_loss < best_loss * 0.999

    _log(f'[Epoch {epoch+1}] saving checkpoints ...')
    _model_sd = model.state_dict()
    _opt_sd = optimizer.state_dict()
    _sched_sd = scheduler.state_dict()
    _scaler_sd = scaler.state_dict() if scaler else {}

    latest_ckpt = copy.deepcopy(checkpoint_template)
    latest_ckpt.update({
        'model_state_dict' : _model_sd,
        'optimizer_state_dict': _opt_sd,
        'scheduler_state_dict': _sched_sd,
        'scaler_state_dict' : _scaler_sd,
        'epoch' : epoch,
        'best_val_loss' : best_loss,
        'no_improve' : no_improve,
        'train_losses' : train_losses.copy(),
        'val_losses' : val_losses.copy(),
    })
    torch.save(latest_ckpt, RESUME_PATH)
    _log(f'[Epoch {epoch+1}] latest checkpoint saved')

    if is_best:
        best_loss = val_loss
        no_improve = 0
        best_ckpt = copy.deepcopy(checkpoint_template)
        best_ckpt.update({
            'model_state_dict' : _model_sd,
            'optimizer_state_dict': _opt_sd,
            'scheduler_state_dict': _sched_sd,
            'scaler_state_dict' : _scaler_sd,
            'epoch' : epoch,
            'best_val_loss' : best_loss,
            'no_improve' : 0,
            'train_losses' : train_losses.copy(),
            'val_losses' : val_losses.copy(),
            'val_metrics' : {k: float(v) for k, v in val_metrics.items()},
        })
        torch.save(best_ckpt, SAVE_PATH)
    else:
        no_improve += 1

    if (epoch + 1) % 5 == 0:
        torch.save(_model_sd,
                   str(BASE_MODELS / f'physfusion_epoch_{epoch+1:02d}.pth'))

    del _model_sd, _opt_sd, _sched_sd, _scaler_sd
    _log(f'[Epoch {epoch+1}] checkpoints saved, state dicts freed  VRAM={_vram_str()}')

    with open(CSV_PATH, 'a', newline='') as f:
        csv.writer(f).writerow([
            epoch+1,
            round(elapsed, 1),
            round(tr_loss, 6),
            round(val_loss, 6),
            round(val_metrics['mae'], 6),
            round(val_metrics['rmse'], 6),
            round(val_metrics['r2'], 6),
            round(val_metrics['nmad'], 6),
            round(val_metrics['outlier_pct'], 4),
            round(val_metrics['within_0.1z'], 4),
            round(val_metrics['within_0.05z'], 4),
            round(val_metrics['bias'], 6),
            round(val_metrics['mean_sigma'], 6),
            round(val_metrics['med_sigma'], 6),
            round(gap, 6),
            round(best_loss, 6),
            no_improve, is_best, current_lr, round(beta_scale, 4),
        ])

    _log(
        f'[Epoch {epoch+1}/{NUM_EPOCHS}] '
        f'loss={tr_loss:.4f}  val={val_loss:.4f}  '
        f'MAE={val_metrics["mae"]:.4f}  NMAD={val_metrics["nmad"]:.4f}  '
        f'R2={val_metrics["r2"]:.4f}  LR={current_lr:.2e}  '
        f'best={best_loss:.4f}  no_imp={no_improve}  '
        f'elapsed={int(time.time()-t0)}s'
    )
    _log(f'[Epoch {epoch+1}] COMPLETE  elapsed={time.time()-t0:.0f}s  VRAM={_vram_str()}')

    if no_improve >= PATIENCE:
        print(f'Early stopping at epoch {epoch+1}')
        break

print('Training complete. Best val_loss:', round(best_loss, 4))


Diagnostic log -> /Users/arpinejanunts/Desktop/Capstone/models/physfusion_diag.txt
Training complete. Best val_loss: -2.5018


## 5. Band Attention Analysis

In [15]:
ckpt = torch.load(SAVE_PATH, map_location=device, weights_only=False)
model.load_state_dict(ckpt["model_state_dict"])
model.eval()
print(f"Best model loaded  (epoch {ckpt.get('epoch','?')+1}, "
      f"val_loss={ckpt.get('best_val_loss','?'):.4f})")

Best model loaded  (epoch 44, val_loss=-2.5018)


In [16]:
with torch.no_grad():
    band_vecs = model.band_embed.embed.weight.cpu().float()  # (5, D)
    band_vecs_n = nn.functional.normalize(band_vecs, dim=1)
    sim_matrix = (band_vecs_n @ band_vecs_n.T).numpy()

fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(sim_matrix, cmap=GALAXY_CMAP, vmin=-1, vmax=1)
ax.set_xticks(range(5)); ax.set_xticklabels(BANDS)
ax.set_yticks(range(5)); ax.set_yticklabels(BANDS)
plt.colorbar(im, ax=ax, label='Cosine similarity')
ax.set_title('Learned Band Embedding Similarity\n(physically informed init, post-training)')
for i in range(5):
    for j in range(5):
        ax.text(j, i, f'{sim_matrix[i,j]:.2f}', ha='center', va='center',
                color='white' if sim_matrix[i,j] < 0.5 else '#222222', fontsize=9)
fig.tight_layout()
out = str(os.path.join(BASE_REPORTS, 'physfusion_band_similarity.png'))
Path(out).parent.mkdir(parents=True, exist_ok=True)
fig.savefig(out)
plt.show()
plt.close(fig)
print(f'Saved -> {out}')

Saved -> /Users/arpinejanunts/Desktop/Capstone/reports/physfusion_band_similarity.png


/var/folders/69/w3lrlv056t9dlrj1sbkzhvyc0000gn/T/ipykernel_8069/2260926990.py:20: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 6. Final Model Evaluation 



In [17]:
all_mu, all_sigma, all_true = [], [], []

cuda = device.type == "cuda"
memfmt = torch.channels_last if cuda else torch.contiguous_format
adev = "cuda" if cuda else "cpu"

with torch.no_grad():
    for imgs, tabs, targets in tqdm(test_loader, desc="Test inference", unit="batch"):
        imgs = imgs.to(device, non_blocking=True, memory_format=memfmt)
        tabs = tabs.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with autocast(adev, enabled=cuda):
            mu, sigma, _, _ = model(imgs, tabs)

        all_mu.append(mu.float().clamp(-0.1, 6.0).cpu().numpy())
        all_sigma.append(sigma.float().clamp(1e-4, 5.0).cpu().numpy())
        all_true.append(targets.cpu().float().numpy())

ymu_full = np.concatenate(all_mu).flatten()
ysigma_full = np.concatenate(all_sigma).flatten()
ytrue_full = np.concatenate(all_true).flatten()

print(f"Full test set  : N={len(ytrue_full):,}  "
      f"NaN in mu={np.isnan(ymu_full).sum()}  "
      f"NaN in sigma={np.isnan(ysigma_full).sum()}")

mask = (ytrue_full >= 0.0) & (ytrue_full <= 2.5) & np.isfinite(ymu_full)
y_pred = ymu_full[mask]
y_true = ytrue_full[mask]
ysigma = ysigma_full[mask]

print(f"Eval subset    : N={mask.sum():,}  "
      f"({mask.mean()*100:.1f}% of test set, z ∈ [0.0, 2.5])\n")

metrics = redshift_metrics(y_pred, y_true)

print("=" * 55)
print(f"  Fusion Model  — Test Metrics  (0.0 ≤ z ≤ 2.5,  N={mask.sum():,})")
print("=" * 55)
for k, v in metrics.items():
    print(f"  {k:<28} {v:.5f}")
print(f"  {'mean_sigma':<28} {ysigma.mean():.5f}")
print(f"  {'median_sigma':<28} {np.median(ysigma):.5f}")
print("=" * 55)

evaluate_by_bin(y_true, y_pred, bins=[0.3, 0.5, 1.0, 1.5, 2.0, 2.5])

Test inference:   0%|          | 0/640 [00:00<?, ?batch/s]

Full test set  : N=40,914  NaN in mu=0  NaN in sigma=0
Eval subset    : N=40,039  (97.9% of test set, z ∈ [0.0, 2.5])

  Fusion Model  — Test Metrics  (0.0 ≤ z ≤ 2.5,  N=40,039)
  mae                          0.05367
  rmse                         0.15636
  r2                           0.87864
  nmad                         0.01905
  bias                         0.00891
  outlier_pct                  3.18190
  within_0.1z                  0.95197
  within_0.05z                 0.88783
  mean_sigma                   0.07590
  median_sigma                 0.03141

Bin                N     RMSE   σ_NMAD      η (%)   η_abs (%)     Bias
--------------------------------------------------------------------
0.3–0.5        7,216   0.1197   0.0160      1.73%       0.42%   0.0023
0.5–1.0       14,815   0.1460   0.0194      2.91%       0.67%  -0.0024
1.0–1.5        2,069   0.3182   0.0573     14.21%       2.61%  -0.0003
1.5–2.0          938   0.3083   0.0602     12.26%       2.13%  -0.0058
2.0–2.5

,Bin,N,RMSE,σ_NMAD,η (%),η_abs (%),Bias
0,0.3–0.5,7216,0.119739,0.016022,1.732262,0.415743,0.002313
1,0.5–1.0,14815,0.145978,0.019441,2.909214,0.668242,-0.002361
2,1.0–1.5,2069,0.318233,0.057303,14.209763,2.609957,-0.000297
3,1.5–2.0,938,0.308254,0.060206,12.260128,2.132196,-0.005826
4,2.0–2.5,1047,0.313931,0.040135,8.500478,2.578797,-0.003269


In [31]:
np.save(os.path.join(BASE_MODELS, "physlatefusion_ytrue.npy"), y_true)
np.save(os.path.join(BASE_MODELS, "physlatefusion_ypred.npy"), y_pred)

## 7. Visualizations

In [18]:
fig = plt.figure(figsize=(16, 12))
gs = gridspec.GridSpec(2, 3, figure=fig, hspace=0.38, wspace=0.32)

dz = (y_pred - y_true) / (1.0 + y_true)

ax0 = fig.add_subplot(gs[0, :2])
try:
    kde = gaussian_kde(np.vstack([y_true, y_pred]), bw_method=0.05)
    z = kde(np.vstack([y_true, y_pred]))
    idx = z.argsort()
    sc = ax0.scatter(y_true[idx], y_pred[idx], c=z[idx],
                      s=1, alpha=0.6, cmap=GALAXY_CMAP, rasterized=True)
    plt.colorbar(sc, ax=ax0, label="Density")
except Exception:
    ax0.scatter(y_true, y_pred, s=1, alpha=0.3,
                color=SPLIT_COLORS["test"], rasterized=True)

ax0.plot([0.0, 2.5], [0.0, 2.5], "k--", lw=1.2, label="Ideal")
ax0.plot([0.0, 2.5],
         [0.15 * (1 + np.array([0.0, 2.5])),
          0.15 * (1 + np.array([0.0, 2.5]))],
         color="gray", lw=0.8, ls=":")
ax0.set_xlabel(r"$z_{\rm spec}$")
ax0.set_ylabel(r"$z_{\rm pred}$")
ax0.set_title(r"$z_{\rm phot}$ vs $z_{\rm spec}$")
ax0.legend(fontsize=9)
ax0.set_xlim(0.0, 2.5)
ax0.set_ylim(-0.1, 2.6)

ax1 = fig.add_subplot(gs[0, 2])
ax1.hist(dz, bins=200, color=SPLIT_COLORS["test"], alpha=0.85)
ax1.axvline(0, color="k", lw=1.2)
ax1.axvline(np.median(dz), color=SPLIT_COLORS["val"],   lw=1.2, ls="--",
            label=f"Median={np.median(dz):.4f}")
ax1.axvline(-0.15, color="gray", lw=0.8, ls=":")
ax1.axvline(+0.15, color="gray", lw=0.8, ls=":")
ax1.set_xlabel(r"$\Delta z / (1+z_{\rm spec})$")
ax1.set_ylabel("Count")
ax1.set_title(r"$\Delta z\,/\,(1+z_{\rm spec})$")
ax1.legend(fontsize=9)

ax2 = fig.add_subplot(gs[1, 0])
ax2.hist(ysigma, bins=150, color=SPLIT_COLORS["val"], alpha=0.85)
ax2.axvline(np.median(ysigma), color="k", lw=1.2, ls="--",
            label=f"Median={np.median(ysigma):.4f}")
ax2.set_xlabel(r"Predicted $\sigma$")
ax2.set_ylabel("Count")
ax2.set_title("Uncertainty distribution")
ax2.legend(fontsize=9)

ax3 = fig.add_subplot(gs[1, 1])
abs_dz = np.abs(dz)
ax3.scatter(ysigma, abs_dz, s=1, alpha=0.15,
            color=SPLIT_COLORS["test"], rasterized=True)
lim = min(float(ysigma.max()), float(abs_dz.max()), 0.5)
ax3.plot([0, lim], [0, lim], "k--", lw=1.0, label="σ = |Δz|")
ax3.set_xlabel(r"Predicted $\sigma$")
ax3.set_ylabel(r"$|\Delta z / (1+z)|$")
ax3.set_title(r"Predicted $\sigma$ vs $|\Delta z|$")
ax3.set_xlim(0, lim)
ax3.set_ylim(0, lim)
ax3.legend(fontsize=9)

ax4 = fig.add_subplot(gs[1, 2])
_bins = [(0.3, 0.5), (0.5, 1.0), (1.0, 1.5), (1.5, 2.0), (2.0, 2.5)]
bin_labels, bin_nmads = [], []
for zlo, zhi in _bins:
    m = (y_true >= zlo) & (y_true < zhi)
    if m.sum() < 10:
        continue
    bm = redshift_metrics(y_pred[m], y_true[m])
    bin_labels.append(f"{zlo:.1f}–{zhi:.1f}")
    bin_nmads.append(bm["nmad"])

x = np.arange(len(bin_labels))
bars = ax4.bar(x, bin_nmads, color=SPLIT_COLORS["test"], alpha=0.85, width=0.5)
ax4.set_xticks(x)
ax4.set_xticklabels(bin_labels, rotation=35, ha="right", fontsize=8)
ax4.set_ylabel("NMAD")
ax4.set_xlabel("Redshift bin")
ax4.set_title("NMAD per redshift bin")
for bar, v in zip(bars, bin_nmads):
    ax4.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.001,
             f"{v:.4f}", ha="center", va="bottom", fontsize=7)

fig.suptitle("Physics-Informed Late Fusion — Test Set Evaluation",
    fontsize=12)

out = os.path.join(BASE_REPORTS, "PhysicallyInformedLateFusion_evaluation_grid.png")
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved → {out}")

Saved → /Users/arpinejanunts/Desktop/Capstone/reports/PhysicallyInformedLateFusion_evaluation_grid.png


/var/folders/69/w3lrlv056t9dlrj1sbkzhvyc0000gn/T/ipykernel_8069/198832974.py:91: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## 8. Calibration Analysis


In [19]:
pit = scipy_norm.cdf((y_true - y_pred) / ysigma)
k_values = np.linspace(0, 3, 100)
coverage = [np.mean(np.abs(y_true - y_pred) <= k * ysigma) for k in k_values]
theory = scipy_norm.cdf(k_values) - scipy_norm.cdf(-k_values)

fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(16, 4),
                                     facecolor="white")
for ax in (ax1, ax2, ax3):
    ax.set_facecolor("white")

fig.suptitle("Physics-Informed Late Fusion — Calibration",
             fontsize=15, fontweight="bold")

# 1: PIT histogram
ax1.hist(pit, bins=50, color=SPLIT_COLORS["train"],
         edgecolor="none", density=True, alpha=0.85)
ax1.axhline(1.0, color="#E9C46A", ls="--", lw=1.5, label="Ideal (uniform)")
ax1.set_xlabel("PIT value")
ax1.set_ylabel("Density")
ax1.set_title("PIT Histogram\n(uniform = well-calibrated)")
ax1.legend(fontsize=9)

# 2: Coverage plot
ax2.plot(theory, coverage, color=SPLIT_COLORS["train"], lw=2, label="Model")
ax2.plot([0, 1], [0, 1], color="#E9C46A", ls="--", lw=1.5,
         label="Perfect calibration")
for ci, col, lab in [(0.683, SPLIT_COLORS["val"],  "68%"),
                     (0.954, SPLIT_COLORS["train"], "95%")]:
    frac = np.mean(np.abs(y_true - y_pred) <=
                   scipy_norm.ppf((1 + ci) / 2) * ysigma)
    ax2.axvline(ci, color=col, ls=":", lw=1,
                label=f"{lab}: obs={frac*100:.1f}%")
ax2.set_xlabel("Expected coverage")
ax2.set_ylabel("Observed coverage")
ax2.set_title("Coverage Plot\n(above diagonal = underconfident)")
ax2.legend(fontsize=9)
ax2.set_xlim(0, 1)
ax2.set_ylim(0, 1)

# 3: σ vs |Δz_norm|
dz_norm = np.abs(y_true - y_pred) / (1.0 + y_true)
ax3.scatter(ysigma, dz_norm, s=2, alpha=0.25,
            color=SPLIT_COLORS["train"], rasterized=True)
order = np.argsort(ysigma)
ws = max(1, len(order) // 30)
med_x, med_y = [], []
for start in range(0, len(order) - ws, ws // 2):
    sl = order[start:start + ws]
    med_x.append(np.median(ysigma[sl]))
    med_y.append(np.median(dz_norm[sl]))
ax3.plot(med_x, med_y, color="#E9C46A", lw=2, zorder=3, label="Running median")
ax3.set_xlabel(r"Predicted $\sigma$")
ax3.set_ylabel(r"$|\Delta z|/(1+z)$")
ax3.set_title("σ vs |Δz|\n(correlated = well-calibrated)")
ax3.legend(fontsize=9)

fig.tight_layout()
out = os.path.join(BASE_REPORTS, "fusion2_calibration.png")
fig.savefig(out, dpi=150, bbox_inches="tight")
plt.show()
plt.close(fig)
print(f"Saved → {out}")

ece = float(np.mean(np.abs(np.array(coverage) - np.array(theory))))
within_1s = np.mean(np.abs(y_true - y_pred) <= ysigma)
within_2s = np.mean(np.abs(y_true - y_pred) <= 2 * ysigma)
print(f"\nCalibration summary")
print(f"  ECE (Expected Calibration Error) : {ece:.4f}  (0 = perfect)")
print(f"  Mean predicted σ                 : {ysigma.mean():.4f}")
print(f"  Fraction within 1σ               : {within_1s*100:.1f}%  (ideal 68.3%)")
print(f"  Fraction within 2σ               : {within_2s*100:.1f}%  (ideal 95.4%)")

Saved → /Users/arpinejanunts/Desktop/Capstone/reports/fusion2_calibration.png

Calibration summary
  ECE (Expected Calibration Error) : 0.0438  (0 = perfect)
  Mean predicted σ                 : 0.0759
  Fraction within 1σ               : 77.7%  (ideal 68.3%)
  Fraction within 2σ               : 96.2%  (ideal 95.4%)


/var/folders/69/w3lrlv056t9dlrj1sbkzhvyc0000gn/T/ipykernel_8069/3406276422.py:60: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()
